# Week 0 Day 2: Advanced Data Cleaning

CariSurg MedTech Pathways, Healthcare AI track.

For Day 2, I cleaned the class columns from Tutorial 2 and then cleaned the Pulse column as my assigned column.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

print(f"Python version: {sys.version}")
print(f"pandas version: {pd.__version__}")


Python version: 3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]
pandas version: 2.3.3


## Load the dataset

I loaded the same reduced emergency triage dataset from Day 1.


In [2]:
candidate_paths = [
    Path("EmergencyTriageDataset_Reduced_Dirty.csv"),
    Path("../data/EmergencyTriageDataset_Reduced_Dirty.csv"),
    Path("../source_materials/week0/EmergencyTriageDataset_Reduced_Dirty.csv"),
    Path("../../../source_materials/week0/EmergencyTriageDataset_Reduced_Dirty.csv"),
    Path("/content/drive/MyDrive/ColabNotebooks/CariSurg_Triage_Test/EmergencyTriageDataset_Reduced_Dirty.csv"),
]

FILE_PATH = next((path for path in candidate_paths if path.exists()), None)
if FILE_PATH is None:
    raise FileNotFoundError("Upload the Week 0 CSV or update FILE_PATH.")

df = pd.read_csv(FILE_PATH)
print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns from {FILE_PATH}")
df.head()


Loaded 2205 rows and 11 columns from ..\source_materials\week0\EmergencyTriageDataset_Reduced_Dirty.csv


,ID,Age,Gender,GCS,SBP,DBP,MAP,pulse,Temp,RR,Fio2
0,1,34,0,15.0,93,67.0,75.67,128.0,36.8,14.0,21.0
1,2,20,Male,15.0,130,90.0,103.33,80.0,37.0,16.0,21.0
2,3,77,Female,14.0,163,105.0,124.33,92.0,36.8,18.0,21.0
3,4,23,0,8.0,100,60.0,73.33,100.0,37.0,12.0,100.0
4,5,86,FEMALE,15.0,150,90.0,110.00,85.0,37.0,19.0,21.0


## Start from the Day 1 Gender cleanup

I cleaned Gender first so the dataset stayed consistent with the Day 1 work.


In [3]:
gender_map = {"male": 1, "female": 0, "1": 1, "0": 0}
df["Gender"] = df["Gender"].astype("string").str.strip().str.lower().map(gender_map).astype("Int64")
print(df["Gender"].value_counts(dropna=False).sort_index())


Gender
0    1029
1    1176
Name: count, dtype: Int64


## Clean GCS

GCS measures level of consciousness. The valid range in the tutorial is 3 to 15. I converted the column to numbers, treated non-numeric values as missing, and used the median to fill missing values because most records are clustered at 15.


In [4]:
df["GCS"] = pd.to_numeric(df["GCS"], errors="coerce")
invalid_gcs = (df["GCS"] < 3) | (df["GCS"] > 15)
print(f"GCS missing after numeric conversion: {df['GCS'].isna().sum()}")
print(f"GCS out-of-range values: {invalid_gcs.sum()}")

df.loc[invalid_gcs, "GCS"] = np.nan
gcs_median = df["GCS"].median()
df["GCS"] = df["GCS"].fillna(gcs_median)

print(f"GCS median used: {gcs_median}")
print(df["GCS"].describe())
print(f"GCS missing after cleaning: {df['GCS'].isna().sum()}")


GCS missing after numeric conversion: 44
GCS out-of-range values: 0
GCS median used: 15.0
count    2205.000000
mean       14.425850
std         1.375031
min         3.000000
25%        15.000000
50%        15.000000
75%        15.000000
max        15.000000
Name: GCS, dtype: float64
GCS missing after cleaning: 0


## Clean SBP

SBP is systolic blood pressure. The valid range in the tutorial is 50 to 250 mmHg. I converted the values to numbers, replaced values outside that range with missing values, and filled them with the median.


In [5]:
df["SBP"] = pd.to_numeric(df["SBP"], errors="coerce")
invalid_sbp = (df["SBP"] < 50) | (df["SBP"] > 250)
print(f"SBP missing after numeric conversion: {df['SBP'].isna().sum()}")
print(f"SBP out-of-range values: {invalid_sbp.sum()}")

df.loc[invalid_sbp, "SBP"] = np.nan
sbp_median = df["SBP"].median()
df["SBP"] = df["SBP"].fillna(sbp_median)

print(f"SBP median used: {sbp_median}")
print(df["SBP"].describe())
print(f"SBP missing after cleaning: {df['SBP'].isna().sum()}")


SBP missing after numeric conversion: 22
SBP out-of-range values: 44
SBP median used: 125.0
count    2205.000000
mean      126.626757
std        26.830227
min        55.000000
25%       110.000000
50%       125.000000
75%       140.000000
max       250.000000
Name: SBP, dtype: float64
SBP missing after cleaning: 0


## Clean Temp

Temp had Celsius values, Celsius strings, and Fahrenheit strings. I converted everything to Celsius first. Then I used the tutorial range of 32.0 to 43.0 degrees Celsius and filled missing values with the median.


In [6]:
def to_celsius(value):
    if pd.isna(value):
        return np.nan

    text = str(value).strip()
    try:
        if text.endswith("C"):
            return float(text[:-1])
        if text.endswith("F"):
            return (float(text[:-1]) - 32) * 5 / 9
        return float(text)
    except ValueError:
        return np.nan

df["Temp"] = df["Temp"].apply(to_celsius)
invalid_temp = (df["Temp"] < 32) | (df["Temp"] > 43)
print(f"Temp missing after conversion: {df['Temp'].isna().sum()}")
print(f"Temp out-of-range values: {invalid_temp.sum()}")

df.loc[invalid_temp, "Temp"] = np.nan
temp_median = round(df["Temp"].median(), 1)
df["Temp"] = df["Temp"].fillna(temp_median)

print(f"Temp median used: {temp_median}")
print(df["Temp"].describe())
print(f"Temp missing after cleaning: {df['Temp'].isna().sum()}")


Temp missing after conversion: 22
Temp out-of-range values: 15
Temp median used: 37.0
count    2205.000000
mean       37.232018
std         0.814011
min        35.000000
25%        37.000000
50%        37.000000
75%        37.400000
max        41.700000
Name: Temp, dtype: float64
Temp missing after cleaning: 0


## Clean Pulse

Pulse was my selected column. It records heart rate in beats per minute. The tutorial valid range is 20 to 250 bpm. I converted non-numeric entries to missing values, replaced impossible pulse values with missing values, and used the median because the column had error values and extreme outliers.


In [7]:
print("Pulse values before cleaning:")
print(df["pulse"].value_counts(dropna=False).head(12))

df["pulse"] = pd.to_numeric(df["pulse"], errors="coerce")
invalid_pulse = (df["pulse"] < 20) | (df["pulse"] > 250)
print(f"Pulse missing after numeric conversion: {df['pulse'].isna().sum()}")
print(f"Pulse out-of-range values: {invalid_pulse.sum()}")

df.loc[invalid_pulse, "pulse"] = np.nan
pulse_median = df["pulse"].median()
df["pulse"] = df["pulse"].fillna(pulse_median)

print(f"Pulse median used: {pulse_median}")
print(df["pulse"].describe())
print(f"Pulse missing after cleaning: {df['pulse'].isna().sum()}")


Pulse values before cleaning:
pulse
80.0     161
100.0    157
90.0     145
110.0    101
85.0      92
120.0     83
88.0      68
70.0      55
95.0      53
84.0      48
82.0      46
105.0     42
Name: count, dtype: int64
Pulse missing after numeric conversion: 44
Pulse out-of-range values: 43
Pulse median used: 90.0
count    2205.000000
mean       94.326984
std        19.881720
min        40.000000
25%        80.000000
50%        90.000000
75%       106.000000
max       170.000000
Name: pulse, dtype: float64
Pulse missing after cleaning: 0


## Final check

After cleaning, GCS, SBP, Temp, and Pulse had no missing values left. Pulse values were kept inside the valid range of 20 to 250 bpm.


In [8]:
checks = df[["GCS", "SBP", "Temp", "pulse"]].isna().sum()
print("Missing values after cleaning:")
print(checks)
print(f"Pulse min after cleaning: {df['pulse'].min()}")
print(f"Pulse max after cleaning: {df['pulse'].max()}")
df[["ID", "GCS", "SBP", "Temp", "pulse"]].head(10)


Missing values after cleaning:
GCS      0
SBP      0
Temp     0
pulse    0
dtype: int64
Pulse min after cleaning: 40.0
Pulse max after cleaning: 170.0


,ID,GCS,SBP,Temp,pulse
0,1,15.0,93.0,36.8,128.0
1,2,15.0,130.0,37.0,80.0
2,3,14.0,163.0,36.8,92.0
3,4,8.0,100.0,37.0,100.0
4,5,15.0,150.0,37.0,85.0
5,6,15.0,100.0,37.0,99.0
6,7,15.0,120.0,37.0,99.0
7,8,15.0,100.0,37.0,85.0
8,9,15.0,110.0,37.0,78.0
9,11,15.0,153.0,37.0,130.0
